In [ ]:
import time
start_time = time.time()

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Selected device: {device}")

model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
model = CrossEncoder(model_name, device=device)
print(f"Loaded model: {model_name}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))

In [ ]:
sentence_pairs = list(zip(dataset["sentence1"], dataset["sentence2"]))
true_labels = np.array(dataset["label"])

batch_size = 64
scores = model.predict(sentence_pairs, batch_size=batch_size, show_progress_bar=True)
scores = np.asarray(scores)
predictions = (scores >= 0.5).astype(int)

print(f"Completed inference for {len(predictions)} examples.")
print(f"Score range: min={float(scores.min()):.4f}, max={float(scores.max()):.4f}")

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": "validation",
        "num_examples": len(dataset),
        "threshold": 0.5,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": device
    }
])

print(results_df.to_string(index=False))

In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["score"] = scores
examples_df["predicted_label"] = predictions
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]
examples_df["sentence1_word_count"] = examples_df["sentence1"].str.split().str.len()
examples_df["sentence2_word_count"] = examples_df["sentence2"].str.split().str.len()
examples_df["avg_word_count"] = ((examples_df["sentence1_word_count"] + examples_df["sentence2_word_count"]) / 2.0).round(2)
examples_df["abs_length_gap"] = (examples_df["sentence1_word_count"] - examples_df["sentence2_word_count"]).abs()

length_bins = [0, 10, 20, 30, 10**9]
length_labels = ["short_0_10", "medium_11_20", "long_21_30", "very_long_31_plus"]
examples_df["length_bucket"] = pd.cut(
    examples_df["avg_word_count"],
    bins=length_bins,
    labels=length_labels,
    include_lowest=True,
    right=True
)

print(examples_df.head(10).to_string(index=False))

In [ ]:
bucket_summary = (
    examples_df.groupby("length_bucket", dropna=False)
    .agg(
        num_examples=("true_label", "size"),
        agreement_rate=("correct", "mean"),
        positive_rate=("predicted_label", "mean"),
        true_positive_rate=("true_label", "mean"),
        avg_score=("score", "mean"),
        avg_sentence1_words=("sentence1_word_count", "mean"),
        avg_sentence2_words=("sentence2_word_count", "mean"),
        avg_abs_length_gap=("abs_length_gap", "mean")
    )
    .reset_index()
)

for col in ["agreement_rate", "positive_rate", "true_positive_rate", "avg_score", "avg_sentence1_words", "avg_sentence2_words", "avg_abs_length_gap"]:
    bucket_summary[col] = bucket_summary[col].astype(float).round(4)

print(bucket_summary.to_string(index=False))

In [ ]:
mismatches_df = examples_df[~examples_df["correct"]].copy()
mismatches_df = mismatches_df.sort_values(by=["length_bucket", "score"], ascending=[True, False])

print(f"Mismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    print(
        mismatches_df[[
            "length_bucket",
            "sentence1_word_count",
            "sentence2_word_count",
            "score",
            "true_label",
            "predicted_label",
            "sentence1",
            "sentence2"
        ]].head(15).to_string(index=False)
    )

In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")